# Avaliação Final — Piloto de Cobrança Preventiva (BemLar)

Este notebook é o **esqueleto** da sua entrega. Os blocos abaixo dizem *o que* precisa ser feito; o código é seu.

Antes de escrever qualquer linha:
1. Leia `00_email_financeiro.md` inteiro.
2. Leia `dicionario.md` inteiro.
3. Abra os cinco CSVs e **olhe os valores**, não só os nomes das colunas.

> A entrega vale mais pelas decisões registradas do que pelo código. Cada bloco marcado com ❓ pede uma resposta escrita — responda em célula markdown, ali mesmo.

---
## Fase 1 — Entendimento do Negócio

❓ Em uma frase, qual é a pergunta de negócio? Qual é a unidade de análise (o que é uma linha)?

❓ O Ricardo fez 5 pedidos no e-mail. Liste os 5 e, para cada um, marque agora sua intenção: **atender**, **atender com ressalva** ou **recusar**. Você pode mudar de ideia depois — mas registre a versão inicial.

1. A pergunta do negócio é: para quais clientes eu devo ligar para que os contratos que possuo não virem prejuízo? No caso da análise, na tabela final uma linha repesentará um cliente da empresa.

2. Pedidos do Ricardo:
    - Maior acurácia possível: Vamos atender, mas tendo em vista que acurácia é uma métrica que nem sempre é confiável, pois ela causa a ilusão de alta performance e pode favorecer classes mais frequentes;
    - Uso do campo "status_contrato": Iremos considerar;
    - Uso das informações de sexo e localização: Não iremos considerar as duas colunas, pois no contexto de treino de modelos, informações sensíveis como essas podem reforçar preconceitos e produzir modelos que carregam estigmas prejudiciais por natureza;
    - Uso do score bureau: Iremos considerar;
    - Uso da lista de motivos de atraso: Iremos considerar.

---
## Fase 2 — Entendimento dos Dados

Carregue os cinco arquivos. Lembre: separador `;`, decimal `,`, datas `dd/mm/aaaa`.

Para **cada** base, responda:
- Qual é a granularidade (o que é uma linha)?
- Quantos contratos ela cobre? Todos, ou só uma parte?
- O que significa um valor vazio nessa base? Sempre a mesma coisa?

❓ Faça um inventário de problemas de qualidade: nulos, valores impossíveis, categorias que deveriam ser a mesma. Registre em uma tabela: problema | onde | quantas linhas | é erro ou artefato | o que você fez.

In [1]:
import pandas as pd
import numpy as np

# Convenções de extração definidas no dicionário de dados:
KW = dict(sep=";", decimal=",", encoding="utf-8")

# 1. Carregamento dos 5 arquivos CSV principais
contratos = pd.read_csv("contratos.csv", **KW)
pagamentos = pd.read_csv("pagamentos.csv", **KW)
compras = pd.read_csv("compras.csv", **KW)
ocorrencias_sac = pd.read_csv("ocorrencias_sac.csv", **KW)
score_bureau = pd.read_csv("score_bureau.csv", **KW)

# 2. Conversão das colunas de data com pd.to_datetime (formato dd/mm/aaaa)
contratos["data_venda"] = pd.to_datetime(contratos["data_venda"], format="%d/%m/%Y")
contratos["data_snapshot"] = pd.to_datetime(contratos["data_snapshot"], format="%d/%m/%Y")

pagamentos["data_vencimento"] = pd.to_datetime(pagamentos["data_vencimento"], format="%d/%m/%Y")
pagamentos["data_pagamento"] = pd.to_datetime(pagamentos["data_pagamento"], format="%d/%m/%Y")

compras["data_compra"] = pd.to_datetime(compras["data_compra"], format="%d/%m/%Y")

ocorrencias_sac["data_ocorrencia"] = pd.to_datetime(ocorrencias_sac["data_ocorrencia"], format="%d/%m/%Y")

score_bureau["data_consulta"] = pd.to_datetime(score_bureau["data_consulta"], format="%d/%m/%Y")

# 3. Diagnóstico e apuração dos 5 DataFrames
total_contratos = contratos["id_contrato"].nunique()
clientes_compras = set(compras["id_cliente"])
contratos_compras = contratos[contratos["id_cliente"].isin(clientes_compras)]["id_contrato"].nunique()

diagnostico = [
    {
        "Base": "contratos.csv",
        "Linhas": len(contratos),
        "Colunas": len(contratos.columns),
        "Granularidade": "1 linha por contrato de crediário",
        "Contratos Cobertos": f"{contratos['id_contrato'].nunique()} ({contratos['id_contrato'].nunique()/total_contratos*100:.1f}%)",
        "Valores Nulos": "0 nulos (100% preenchido)"
    },
    {
        "Base": "pagamentos.csv",
        "Linhas": len(pagamentos),
        "Colunas": len(pagamentos.columns),
        "Granularidade": "1 linha por parcela vencida até o snapshot",
        "Contratos Cobertos": f"{pagamentos['id_contrato'].nunique()} ({pagamentos['id_contrato'].nunique()/total_contratos*100:.1f}%)",
        "Valores Nulos": f"data_pagamento: {pagamentos['data_pagamento'].isnull().sum()} ({pagamentos['data_pagamento'].isnull().mean()*100:.2f}%) [parcela não paga]"
    },
    {
        "Base": "compras.csv",
        "Linhas": len(compras),
        "Colunas": len(compras.columns),
        "Granularidade": "1 linha por compra adicional do cliente na rede",
        "Contratos Cobertos": f"{contratos_compras} ({contratos_compras/total_contratos*100:.1f}% via id_cliente)",
        "Valores Nulos": "0 nulos no arquivo (379 contratos s/ compras adicionais)"
    },
    {
        "Base": "ocorrencias_sac.csv",
        "Linhas": len(ocorrencias_sac),
        "Colunas": len(ocorrencias_sac.columns),
        "Granularidade": "1 linha por atendimento/chamado no SAC",
        "Contratos Cobertos": f"{ocorrencias_sac['id_contrato'].nunique()} ({ocorrencias_sac['id_contrato'].nunique()/total_contratos*100:.1f}%)",
        "Valores Nulos": "0 nulos no arquivo (1.011 contratos sem SAC)"
    },
    {
        "Base": "score_bureau.csv",
        "Linhas": len(score_bureau),
        "Colunas": len(score_bureau.columns),
        "Granularidade": "1 linha por consulta de bureau de crédito",
        "Contratos Cobertos": f"{score_bureau['id_contrato'].nunique()} ({score_bureau['id_contrato'].nunique()/total_contratos*100:.1f}%)",
        "Valores Nulos": "0 nulos (100% preenchido)"
    }
]

df_diagnostico = pd.DataFrame(diagnostico)
df_diagnostico


Base,Linhas,Colunas,Granularidade,Contratos Cobertos,Valores Nulos
contratos.csv,3000,14,1 linha por contrato de crediário,3000 (100.0%),0 nulos (100% preenchido)
pagamentos.csv,28861,5,1 linha por parcela vencida até o snapshot,3000 (100.0%),data_pagamento: 1690 (5.86%) [parcela não paga]
compras.csv,3983,4,1 linha por compra adicional do cliente na rede,2621 (87.4% via id_cliente),0 nulos no arquivo (379 contratos s/ compras adicionais)
ocorrencias_sac.csv,3446,4,1 linha por atendimento/chamado no SAC,1989 (66.3%),0 nulos no arquivo (1.011 contratos sem SAC)
score_bureau.csv,4490,4,1 linha por consulta de bureau de crédito,3000 (100.0%),0 nulos (100% preenchido)


### Respostas às perguntas da Fase 2 — Entendimento dos Dados

#### 1. `contratos.csv`
- **Granularidade:** Uma linha por **contrato de crediário**. A chave primária única é `id_contrato`.
- **Cobertura de contratos:** Cobre **3.000 contratos (100% da base)**. Abrange 1.824 clientes únicos (`id_cliente`).
- **Apuração de valores nulos:** **0 valores nulos** em todas as 14 colunas. O cadastro e as condições de contratação estão completamente preenchidos.

---

#### 2. `pagamentos.csv`
- **Granularidade:** Uma linha por **parcela já vencida até a data do snapshot (15/03/2026)**. Chave composta por `(id_contrato, n_parcela)`.
- **Cobertura de contratos:** Cobre **3.000 contratos (100% da base)**. Todo contrato do cadastro possui ao menos uma parcela vencida registrada (total de 28.861 parcelas).
- **Apuração de valores nulos:** Apenas a coluna `data_pagamento` possui valores vazios: **1.690 valores nulos** (5,86% do total de parcelas).
  - *O que significa o valor vazio?* **É um artefato de negócio, não um erro:** indica que a parcela já venceu e **ainda não foi paga** até a data do corte (15/03/2026). As demais 4 colunas não têm valores nulos.

---

#### 3. `compras.csv`
- **Granularidade:** Uma linha por **compra adicional realizada pelo cliente na rede** (à vista ou em outro crediário). A chave de ligação é `id_cliente` (e não `id_contrato`).
- **Cobertura de contratos:** Cobre **1.493 clientes únicos (81,9% dos 1.824 clientes)**, correspondendo a **2.621 contratos (87,4% dos 3.000 contratos)**. Há **379 contratos (12,6%)** cujos clientes não possuem outras compras registradas nessa base.
- **Apuração de valores nulos:** **0 valores nulos** no arquivo bruto (4 colunas preenchidas).
  - *O que significa a ausência na junção?* Ao cruzar com a base de contratos via `id_cliente`, os 379 contratos sem compras geram nulos que significam **ausência de evento** (o cliente não efetuou compras extras na rede). Na engenharia de features, devem ser tratados com zero (`qtd_compras = 0`, `total_gasto = 0.0`), e não como dados faltantes.

---

#### 4. `ocorrencias_sac.csv`
- **Granularidade:** Uma linha por **atendimento/ocorrência registrada no SAC** vinculada a um contrato.
- **Cobertura de contratos:** Cobre **1.989 contratos únicos (66,3% dos 3.000 contratos)**. Um total de **1.011 contratos (33,7%)** não possui nenhuma ocorrência registrada no SAC.
- **Apuração de valores nulos:** **0 valores nulos** no arquivo bruto.
  - *O que significa a ausência na junção?* Os 1.011 contratos sem registro refletem **ausência de evento** (o cliente nunca acionou o SAC). Na agregação de variáveis preditivas, ausência de contato vira contagem 0 de ocorrências.

---

#### 5. `score_bureau.csv`
- **Granularidade:** Uma linha por **consulta de bureau de crédito (Serasa/SPC)** referente ao contrato. Um mesmo contrato pode ter mais de uma consulta em momentos diferentes (são 4.490 consultas para 3.000 contratos).
- **Cobertura de contratos:** Cobre **3.000 contratos (100% da base)**. Todos os contratos possuem ao menos uma consulta registrada.
- **Apuração de valores nulos:** **0 valores nulos** em todas as 4 colunas.
  - *Atenção metodológica:* Como há múltiplos registros por contrato, ao criar features é mandatório filtrar consultas realizadas **até a data de referência** de cada contrato (`filtra_por_data`), evitando vazamento temporal (*data leakage*).


In [2]:
# Inventário de qualidade de dados: nulos, valores impossíveis e categorias inconsistentes

inventario = [
    {
        "Problema": "Valores nulos em data de pagamento",
        "Onde": "pagamentos.csv (data_pagamento)",
        "Quantas linhas": f"{pagamentos['data_pagamento'].isnull().sum()} parcelas ({pagamentos['data_pagamento'].isnull().mean()*100:.2f}%)",
        "É erro ou artefato?": "Artefato de negócio",
        "O que você fez": "Mantido nulo no histórico; usado para calcular o alvo (inadimplência) e features de atraso."
    },
    {
        "Problema": "Idade biologicamente impossível (> 100 anos)",
        "Onde": "contratos.csv (idade)",
        "Quantas linhas": f"{(contratos['idade'] > 100).sum()} contratos (ex.: idades 108 e 137 anos)",
        "É erro ou artefato?": "Erro cadastral (digitação)",
        "O que você fez": "Imputar pela mediana de idade dos contratos ou tratar na auditoria de features."
    },
    {
        "Problema": "Renda declarada igual a R$ 0,00",
        "Onde": "contratos.csv (renda_declarada)",
        "Quantas linhas": f"{(contratos['renda_declarada'] == 0).sum()} contratos",
        "É erro ou artefato?": "Dado suspeito / sem comprovação formal",
        "O que você fez": "Avaliar se reflete desemprego/informalidade ou imputar com a mediana."
    },
    {
        "Problema": "Categoria quase duplicada de bairro",
        "Onde": "contratos.csv (bairro)",
        "Quantas linhas": f"{(contratos['bairro'] == 'Jd. America').sum()} linhas 'Jd. America' vs {(contratos['bairro'] == 'Jardim América').sum()} 'Jardim América'",
        "É erro ou artefato?": "Erro de digitação / falta de padronização",
        "O que você fez": "Padronizar 'Jd. America' para 'Jardim América' (caso bairro seja utilizado, respeitando a auditoria ética)."
    },
    {
        "Problema": "Ausência de histórico de compras adicionais",
        "Onde": "compras.csv (merge via id_cliente)",
        "Quantas linhas": f"{total_contratos - contratos_compras} contratos ({(total_contratos - contratos_compras)/total_contratos*100:.2f}%)",
        "É erro ou artefato?": "Ausência de evento",
        "O que você fez": "Preencher contagem de compras e volume financeiro com 0 no merge."
    },
    {
        "Problema": "Ausência de chamados de atendimento",
        "Onde": "ocorrencias_sac.csv (merge via id_contrato)",
        "Quantas linhas": f"{total_contratos - ocorrencias_sac['id_contrato'].nunique()} contratos ({(total_contratos - ocorrencias_sac['id_contrato'].nunique())/total_contratos*100:.2f}%)",
        "É erro ou artefato?": "Ausência de evento",
        "O que você fez": "Preencher contagem de chamados com 0 no merge."
    },
    {
        "Problema": "Ausência de chaves de identificação",
        "Onde": "motivos_atraso.csv",
        "Quantas linhas": "1.019 linhas",
        "É erro ou artefato?": "Limitação de extração",
        "O que você fez": "Não incluir na modelagem tabular por impossibilidade de cruzamento com contratos."
    }
]

df_inventario = pd.DataFrame(inventario)
df_inventario


Problema,Onde,Quantas linhas,É erro ou artefato?,O que você fez
Valores nulos em data de pagamento,pagamentos.csv (data_pagamento),1690 parcelas (5.86%),Artefato de negócio,Mantido nulo no histórico; usado para calcular o alvo (inadimplência) e features de atraso.
Idade biologicamente impossível (> 100 anos),contratos.csv (idade),3 contratos (ex.: idades 108 e 137 anos),Erro cadastral (digitação),Imputar pela mediana de idade dos contratos ou tratar na auditoria de features.
"Renda declarada igual a R$ 0,00",contratos.csv (renda_declarada),4 contratos,Dado suspeito / sem comprovação formal,Avaliar se reflete desemprego/informalidade ou imputar com a mediana.
Categoria quase duplicada de bairro,contratos.csv (bairro),232 linhas 'Jd. America' vs 429 'Jardim América',Erro de digitação / falta de padronização,"Padronizar 'Jd. America' para 'Jardim América' (caso bairro seja utilizado, respeitando a auditoria ética)."
Ausência de histórico de compras adicionais,compras.csv (merge via id_cliente),379 contratos (12.63%),Ausência de evento,Preencher contagem de compras e volume financeiro com 0 no merge.
Ausência de chamados de atendimento,ocorrencias_sac.csv (merge via id_contrato),1011 contratos (33.70%),Ausência de evento,Preencher contagem de chamados com 0 no merge.
Ausência de chaves de identificação,motivos_atraso.csv,1.019 linhas,Limitação de extração,Não incluir na modelagem tabular por impossibilidade de cruzamento com contratos.


---
## Fase 3 — Preparação dos Dados

### 3.1 Construir o alvo

A regra está no enunciado e no dicionário. Você precisa, para cada contrato:
1. identificar a **parcela de referência**;
2. calcular a **data de referência**;
3. derivar `inadimplente_30d`.

❓ Depois de construir: qual é a prevalência de positivos? Ela é compatível com o que o Ricardo descreveu no e-mail?

❓ Compare o seu alvo com `status_contrato`. Eles concordam? Onde discordam, quem está certo — e por quê isso importa para a definição do que você vai prever?

In [ ]:
# TODO: identificar a parcela de referencia por contrato
# TODO: calcular data_referencia
# TODO: construir inadimplente_30d


### 3.2 A função de corte temporal

Está pronta. Use em **toda** base de eventos antes de agregar qualquer coisa.

In [ ]:
def filtra_por_data(df_eventos, chave, col_data, datas_ref, col_ref="data_referencia"):
    """Mantem apenas os eventos ocorridos ATE a data de referencia de cada contrato.

    df_eventos : DataFrame de eventos (pagamentos, sac, compras, consultas...)
    chave      : coluna de juncao com datas_ref (ex.: "id_contrato")
    col_data   : coluna de data do evento no df_eventos
    datas_ref  : DataFrame com [chave, col_ref]
    """
    out = df_eventos.merge(datas_ref[[chave, col_ref]], on=chave, how="inner")
    out = out[out[col_data] <= out[col_ref]]
    return out.drop(columns=[col_ref])


### 3.3 Construir as features

Uma linha por contrato. Para cada base de eventos, decida a agregação **pelo significado**: contagem é frequência, soma é volume, média é intensidade.

❓ Para cada base, escreva antes de codar: *qual comportamento essa agregação está tentando capturar?*

❓ Ausência de evento vira 0, nulo, ou mediana? A resposta é a mesma para toda coluna?

Encapsule tudo em uma função — ela precisa rodar do zero, a partir dos CSVs originais:

```python
def construir_features(caminho_dados, datas_referencia):
    ...
    return df
```

In [ ]:
# TODO: construir_features()


### 3.4 Auditoria de colunas

Antes de fechar o `features.csv`, passe **cada coluna** pelas três perguntas:

1. **Essa informação existiria no momento em que a previsão seria feita?**
2. **É uma variável protegida ou sensível?**
3. **Descreve comportamento, ou só descreve quem a pessoa é?**

❓ Monte a tabela: coluna | decisão (entra / sai) | justificativa. Toda coluna do arquivo precisa aparecer, inclusive as que você descartou.

Salve o resultado em `features.csv`.

In [ ]:
# TODO: auditoria + gravar features.csv


---
## Fase 4 — Modelagem

- Split **estratificado**.
- Um **baseline simples** (árvore de decisão) e pelo menos mais um modelo.
- Depois, refaça com **split temporal**: ordene pela data de referência, 75% mais antigos treinam, 25% mais recentes testam, sem embaralhar.

❓ O resultado do split temporal foi diferente do aleatório? O que isso diz — e o que **não** diz?

In [ ]:
# TODO: split estratificado + baseline + segundo modelo


In [ ]:
# TODO: split temporal e comparacao


---
## Fase 5 — Avaliação

A equipe faz **80 ligações**. A avaliação acontece nesse corte, não no threshold de 0,50.

1. Ordene o conjunto de teste pela probabilidade prevista (`predict_proba`).
2. Corte nos 80 primeiros.
3. Calcule **precision** e **recall** nesse recorte.
4. Compare com pelo menos uma **fila-base burra**: aleatória, ou ordenada por valor da parcela.

❓ Sua fila é melhor que a fila-base? Quanto? Se a diferença for pequena, diga isso — é um resultado legítimo.

❓ Traduza para reais: quanto o piloto evita de prejuízo por semana, e quantas ligações são desperdiçadas para isso?

❓ Você prioriza **precision** ou **recall** neste caso? Justifique pelo negócio, não pela métrica.

In [ ]:
# TODO: precision@80, recall@80, fila-base, comparacao


In [ ]:
# TODO: importancia de variaveis - e cuidado ao interpretar


### Gerar a fila

Salve `fila_80.csv` com `id_contrato` e `probabilidade`, ordenado do maior risco para o menor.

In [ ]:
# TODO: gerar fila_80.csv


---
## Fechamento

Antes de enviar, confira o checklist da seção 9 do enunciado.

Faltam ainda dois arquivos que **não** são gerados aqui:
- `decisoes.md` — o que entrou, o que saiu, cada pedido do Ricardo respondido, e o model card.
- `apresentacao.pptx` — 3 slides, para o Ricardo, não para a banca.

> Se algum resultado ficou bom demais, pergunte: *que outra coisa poderia produzir esse mesmo número?*